# Exercise 6

In [ ]:
from functions import RandomnessTests
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import random
random.seed(42)
import time
import tracemalloc
np.random.seed(42)
rng = np.random.default_rng(42)
from scipy.special import factorial
from scipy import stats
import heapq
from scipy.stats import chisquare

## Part 1

In [ ]:
random.seed(42)

m = 10
A = 1.0
n_samples = 10000
burn_in = 1000

def g(i):
    return A**i / np.math.factorial(i)

def mh_independence(num_samples=10000, burn_in=1000):
    samples = []
    x = np.random.randint(0, m+1)

    for t in range(num_samples + burn_in):
        j = np.random.randint(0, m+1)

        alpha = min(1, g(j) / g(x))
        if np.random.rand() < alpha:
            x = j

        if t >= burn_in:
            samples.append(x)

    return np.array(samples)

samples = mh_independence(n_samples, burn_in)

counts = np.bincount(samples, minlength=m+1)
empirical = counts / counts.sum()

true_p = np.array([g(i) for i in range(m+1)])
true_p /= true_p.sum()

chi2_stat, p_value = chisquare(counts, f_exp=true_p * counts.sum())

print(empirical)
print(true_p)
print(p_value)

## Part 2

In [ ]:
random.seed(42)

A1 = 4
A2 = 4
m = 10
n_samples = 10000

def g(i, j):
    if i < 0 or j < 0 or i + j > m:
        return 0
    return (A1**i / factorial(i)) * (A2**j / factorial(j))

### a

In [ ]:
random.seed(42)

def mh_direct(N):
    samples = []

    i, j = 0, 0

    for _ in range(N):

        move = np.random.randint(4)

        if move == 0:
            ip, jp = i + 1, j
        elif move == 1:
            ip, jp = i - 1, j
        elif move == 2:
            ip, jp = i, j + 1
        else:
            ip, jp = i, j - 1

        if g(ip, jp) > 0:

            alpha = min(1, g(ip, jp)/g(i, j))

            if np.random.rand() < alpha:
                i, j = ip, jp

        samples.append((i, j))

    return np.array(samples)

In [ ]:
random.seed(42)
samples_a = mh_direct(n_samples)

### b

In [ ]:
random.seed(42)
def coord_mh(N):

    samples = []

    i, j = 0, 0

    for _ in range(N):

        ip = i + np.random.choice([-1, 1])

        if g(ip, j) > 0:
            alpha = min(1, g(ip, j)/g(i, j))

            if np.random.rand() < alpha:
                i = ip

        jp = j + np.random.choice([-1, 1])

        if g(i, jp) > 0:
            alpha = min(1, g(i, jp)/g(i, j))

            if np.random.rand() < alpha:
                j = jp

        samples.append((i, j))

    return np.array(samples)


In [ ]:
random.seed(42)
samples_b = coord_mh(n_samples)

### c

In [ ]:
random.seed(42)
def sample_i_given_j(j):

    values = np.arange(m-j+1)

    probs = np.array([
        A1**i / factorial(i)
        for i in values
    ])

    probs /= probs.sum()

    return np.random.choice(values, p=probs)


def sample_j_given_i(i):

    values = np.arange(m-i+1)

    probs = np.array([
        A2**j / factorial(j)
        for j in values
    ])

    probs /= probs.sum()

    return np.random.choice(values, p=probs)

def gibbs(N):

    samples = []

    i, j = 0, 0

    for _ in range(N):

        i = sample_i_given_j(j)
        j = sample_j_given_i(i)

        samples.append((i, j))

    return np.array(samples)

In [ ]:
random.seed(42)
samples_c = gibbs(n_samples)

In [ ]:
random.seed(42)

states = []

weights = []

for i in range(m+1):
    for j in range(m+1-i):
        states.append((i,j))
        weights.append(g(i,j))

weights = np.array(weights)
weights /= weights.sum()

In [ ]:
random.seed(42)
def chi2_test(samples):

    burnin = 1000
    samples = samples[burnin:]

    observed = []

    for state in states:
        observed.append(
            np.sum(
                np.all(samples == state, axis=1)
            )
        )

    observed = np.array(observed)

    expected = len(samples) * weights

    chi2, p = chisquare(observed, expected)

    return chi2, p

In [ ]:
random.seed(42)

print(f"Direct MH: chi2={chi2_test(samples_a)[0]:.4f}, p={chi2_test(samples_a)[1]:.4f}")
print(f"Coordinate MH: {chi2_test(samples_b)[0]:.4f}, p={chi2_test(samples_b)[1]:.4f}")
print(f"Gibbs: {chi2_test(samples_c)[0]:.4f}, p={chi2_test(samples_c)[1]:.4f}")

## Part 3

### a

In [ ]:
random.seed(42)
rho = 0.5

Sigma = np.array([
    [1, rho],
    [rho, 1]
])

xi, gamma = np.random.multivariate_normal(
    mean=[0,0],
    cov=Sigma
)

theta = np.exp(xi)
psi = np.exp(gamma)

print(f'theta: {theta:.4f}, psi: {psi:.4f}')

### b

In [ ]:
random.seed(42)
n = 10

x = np.random.normal(
    loc=theta,
    scale=np.sqrt(psi),
    size=n
)

print(x)

### d

In [ ]:
random.seed(42)
rho = 0.5

def log_posterior(theta, psi, data):

    if theta <= 0 or psi <= 0:
        return -np.inf

    n = len(data)

    SSE = np.sum((data - theta)**2)

    loglik = (
        -n/2*np.log(psi)
        -SSE/(2*psi)
    )

    logprior = (
        -np.log(theta)
        -np.log(psi)
        -(
            np.log(theta)**2
            -2*rho*np.log(theta)*np.log(psi)
            +np.log(psi)**2
          )
        /(2*(1-rho**2))
    )

    return loglik + logprior

def metropolis(data,
               N=50000,
               sigma_theta=0.2,
               sigma_psi=0.2):

    theta = np.mean(data)
    psi = np.var(data)

    samples = np.zeros((N,2))

    current = log_posterior(
        theta,
        psi,
        data
    )

    for k in range(N):

        theta_prop = theta*np.exp(
            sigma_theta*np.random.randn()
        )

        psi_prop = psi*np.exp(
            sigma_psi*np.random.randn()
        )

        proposed = log_posterior(
            theta_prop,
            psi_prop,
            data
        )

        alpha = np.exp(
            proposed-current
        )

        if np.random.rand() < min(1, alpha):

            theta = theta_prop
            psi = psi_prop
            current = proposed

        samples[k] = [theta, psi]

    return samples

In [ ]:
random.seed(42)
samples = metropolis(x)

In [ ]:
random.seed(42)
burnin = 5000

theta_hat = np.mean(samples[burnin:,0])
psi_hat = np.mean(samples[burnin:,1])

print(f'Posterior mean estimates - theta: {theta_hat:.4f}, psi: {psi_hat:.4f}')

### e

In [ ]:
random.seed(42)
n = [100, 1000]
for i, ni in enumerate(n):
    plt.subplot(1, len(n), i+1)
    samples_loop = metropolis(x[:ni])
    theta_hat_loop = np.mean(samples_loop[burnin:,0])
    plt.hist(samples_loop[:,0], bins=50, density=True, color="blue", label="Posterior samples")
    plt.axvline(theta_hat_loop, linestyle="--", color="red", label=f"Estimated theta (n={ni})")
    plt.title(f"Posterior of theta (n={ni})")
    plt.xlabel("theta")
    plt.ylabel("Density")
    plt.legend()
plt.show()